# SkinMeta DermaAI — Product Bridge Layer
## Connects recommender ingredient output → real buyable products

| Section | Description |
|---|---|
| 1 | Load & inspect product dataset |
| 2 | Filter acne-relevant products |
| 3 | Parse ingredient lists |
| 4 | Add acne type tags |
| 5 | Build the ingredient matcher |
| 6 | Full recommendation pipeline with explanations |
| 7 | Save outputs |

In [2]:
# ─────────────────────────────────────────────
# SECTION 1 — Load & Inspect Product Dataset
# Upload skincare_products.csv from Kaggle first
# Dataset: https://www.kaggle.com/datasets/eward96/skincare-products-clean-dataset
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import json
import warnings
warnings.filterwarnings('ignore')

# Load the product dataset
products = pd.read_csv('skincare_products.csv')

print('Shape:', products.shape)
print('\nColumns:', products.columns.tolist())
print('\nSample rows:')
display(products.head(3))
print('\nProduct types available:')
print(products['product_type'].value_counts() if 'product_type' in products.columns else 'No product_type column found')
print('\nNull counts:')
print(products.isnull().sum())

Shape: (1138, 5)

Columns: ['product_name', 'product_url', 'product_type', 'clean_ingreds', 'price']

Sample rows:


,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",£5.20
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",£13.00
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",£6.20



Product types available:
product_type
Mask           124
Body Wash      123
Moisturiser    115
Cleanser       115
Serum          113
Eye Care       100
Mist            80
Oil             76
Toner           73
Balm            61
Exfoliator      57
Bath Salts      36
Bath Oil        33
Peel            32
Name: count, dtype: int64

Null counts:
product_name     0
product_url      0
product_type     0
clean_ingreds    0
price            0
dtype: int64


In [8]:
# ─────────────────────────────────────────────
# SECTION 2 — Filter for Acne-Relevant Products
#
# Strategy: keep any product whose ingredient list
# contains at least ONE acne-fighting ingredient.
# ─────────────────────────────────────────────

# These are the key acne-fighting ingredients from your dermatology CSV
ACNE_INGREDIENTS = [
    'salicylic acid',
    'benzoyl peroxide',
    'niacinamide',
    'azelaic acid',
    'zinc pca',
    'zinc gluconate',
    'retinol',
    'retinyl palmitate',
    'adapalene',
    'glycolic acid',
    'lactic acid',
    'mandelic acid',
    'sulfur',
    'tea tree',
    'witch hazel',
    'hyaluronic acid',   # used in acne routines for hydration
    'caffeine',          # for dark circles concern
    'vitamin c',
    'ascorbic acid',
    'kojic acid',
    'alpha arbutin',
    'tranexamic acid',
]

def contains_acne_ingredient(ingredients_str):
    """Return True if the ingredient string contains any acne-fighting ingredient."""
    if pd.isna(ingredients_str):
        return False
    ing_lower = str(ingredients_str).lower()
    return any(acne_ing in ing_lower for acne_ing in ACNE_INGREDIENTS)

# Find the most likely ingredients column robustly
# (common variants: ingredients, ingredient_list, inci_list, clean_ingreds, etc.)
preferred_exact = [
    'ingredients', 'ingredient', 'ingredient_list', 'ingredients_list',
    'inci', 'inci_list', 'full_ingredients', 'clean_ingreds'
 ]
normalized_to_original = {c.strip().lower(): c for c in products.columns}
ing_col = None

for name in preferred_exact:
    if name in normalized_to_original:
        ing_col = normalized_to_original[name]
        break

if ing_col is None:
    candidates = [
        c for c in products.columns
        if any(k in c.lower() for k in ['ingredient', 'ingred', 'inci'])
    ]
    if candidates:
        ing_col = candidates[0]

if ing_col is None:
    print('ERROR: Could not find an ingredients column.')
    print('Available columns:')
    print(products.columns.tolist())
    raise KeyError('No ingredient-like column found. Set ing_col manually to a valid column name.')
else:
    print(f'Using ingredients column: "{ing_col}"')

# Apply filter
mask = products[ing_col].apply(contains_acne_ingredient)
acne_products = products[mask].copy().reset_index(drop=True)

print(f'\nOriginal products  : {len(products)}')
print(f'Acne-relevant      : {len(acne_products)}')
print(f'Filtered out       : {len(products) - len(acne_products)}')

# Which acne ingredients were found most?
print('\nAcne ingredient frequency in filtered dataset:')
for ing in ACNE_INGREDIENTS:
    count = acne_products[ing_col].str.lower().str.contains(ing, na=False).sum()
    if count > 0:
        print(f'  {ing:<30} : {count} products')

Using ingredients column: "clean_ingreds"

Original products  : 1138
Acne-relevant      : 382
Filtered out       : 756

Acne ingredient frequency in filtered dataset:
  salicylic acid                 : 85 products
  niacinamide                    : 68 products
  azelaic acid                   : 1 products
  zinc pca                       : 17 products
  zinc gluconate                 : 22 products
  retinol                        : 22 products
  retinyl palmitate              : 30 products
  glycolic acid                  : 34 products
  lactic acid                    : 100 products
  mandelic acid                  : 9 products
  sulfur                         : 1 products
  caffeine                       : 101 products
  ascorbic acid                  : 43 products
  tranexamic acid                : 2 products


In [9]:
# ─────────────────────────────────────────────
# SECTION 3 — Parse & Standardize Ingredient Lists
#
# The raw ingredient column is one long string like:
# "Aqua, Niacinamide, Salicylic Acid, Glycerin..."
# We parse it into a clean list for matching.
# ─────────────────────────────────────────────

def parse_ingredients(ingredients_str):
    """
    Parse raw ingredient string into a cleaned list.
    Handles comma-separated, semicolon-separated formats.
    Returns list of lowercase stripped ingredient names.
    """
    if pd.isna(ingredients_str):
        return []
    # Split on commas or semicolons
    raw = re.split(r'[,;]', str(ingredients_str))
    cleaned = []
    for item in raw:
        item = item.strip().lower()
        # Remove percentage annotations like (2%)
        item = re.sub(r'\(.*?\)', '', item).strip()
        # Remove asterisks, dots, slashes
        item = re.sub(r'[*./\\]', '', item).strip()
        if item and len(item) > 1:
            cleaned.append(item)
    return cleaned

def get_acne_ingredients_present(ingredients_list):
    """Return which acne-fighting ingredients are in this product."""
    present = []
    for acne_ing in ACNE_INGREDIENTS:
        for product_ing in ingredients_list:
            if acne_ing in product_ing:
                present.append(acne_ing)
                break
    return present

# Parse ingredient lists
acne_products['ingredients_list'] = acne_products[ing_col].apply(parse_ingredients)
acne_products['acne_ingredients_present'] = acne_products['ingredients_list'].apply(
    get_acne_ingredients_present
)
acne_products['acne_ingredient_count'] = acne_products['acne_ingredients_present'].apply(len)

# Clean price column (remove £, $, convert to float)
if 'price' in acne_products.columns:
    acne_products['price_clean'] = (
        acne_products['price']
        .astype(str)
        .str.replace(r'[£$€,]', '', regex=True)
        .str.strip()
    )
    acne_products['price_clean'] = pd.to_numeric(acne_products['price_clean'], errors='coerce')

print('Ingredient parsing complete.')
print('\nSample parsed products:')
display(acne_products[['product_type' if 'product_type' in acne_products.columns else acne_products.columns[0],
                        'acne_ingredients_present',
                        'acne_ingredient_count']].head(8))

print(f'\nProducts with 2+ acne ingredients: {(acne_products["acne_ingredient_count"] >= 2).sum()}')
print(f'Products with 3+ acne ingredients: {(acne_products["acne_ingredient_count"] >= 3).sum()}')

Ingredient parsing complete.

Sample parsed products:


,product_type,acne_ingredients_present,acne_ingredient_count
0,Moisturiser,[niacinamide],1
1,Moisturiser,[lactic acid],1
2,Moisturiser,[niacinamide],1
3,Moisturiser,"[salicylic acid, niacinamide]",2
4,Moisturiser,[caffeine],1
5,Moisturiser,[caffeine],1
6,Moisturiser,[niacinamide],1
7,Moisturiser,[caffeine],1



Products with 2+ acne ingredients: 113
Products with 3+ acne ingredients: 33


In [10]:
# ─────────────────────────────────────────────
# SECTION 4 — Auto-Tag Acne Types
#
# Instead of manual tagging, we use ingredient-based
# rules to automatically assign acne_type tags.
#
# Rule logic mirrors what your dermatology CSV encodes:
#   Comedonal  → salicylic acid, zinc pca
#   Inflammatory → benzoyl peroxide, niacinamide, azelaic acid
#   Cystic     → benzoyl peroxide, azelaic acid (high potency)
#   Fungal     → zinc pca, azelaic acid
#   Pigmentation → niacinamide, vitamin c, kojic acid, alpha arbutin
#   Dark Circles → caffeine, vitamin c, hyaluronic acid
# ─────────────────────────────────────────────

ACNE_TYPE_RULES = {
    'Comedonal': ['salicylic acid', 'zinc pca', 'glycolic acid', 'lactic acid'],
    'Inflammatory': ['benzoyl peroxide', 'niacinamide', 'azelaic acid', 'tea tree'],
    'Cystic': ['benzoyl peroxide', 'azelaic acid', 'retinol', 'adapalene'],
    'Fungal': ['zinc pca', 'zinc gluconate', 'azelaic acid', 'sulfur'],
    'Pigmentation': ['niacinamide', 'vitamin c', 'ascorbic acid', 'kojic acid',
                     'alpha arbutin', 'tranexamic acid'],
    'Dark Circles': ['caffeine', 'vitamin c', 'hyaluronic acid'],
}

def assign_acne_types(acne_ings_present):
    """
    Given the list of acne ingredients present in a product,
    return all acne types this product is suitable for.
    A product can match multiple types.
    """
    matched_types = []
    for acne_type, rule_ingredients in ACNE_TYPE_RULES.items():
        # Product matches this type if it contains ANY of the rule ingredients
        if any(ing in acne_ings_present for ing in rule_ingredients):
            matched_types.append(acne_type)
    return matched_types if matched_types else ['General']

acne_products['acne_types'] = acne_products['acne_ingredients_present'].apply(assign_acne_types)
acne_products['acne_types_str'] = acne_products['acne_types'].apply(lambda x: ', '.join(x))

print('Acne type tagging complete.')
print('\nAcne type distribution:')
from collections import Counter
all_types = [t for types in acne_products['acne_types'] for t in types]
for t, count in sorted(Counter(all_types).items(), key=lambda x: -x[1]):
    print(f'  {t:<20}: {count} products')

print('\nSample tagged products:')
cols_to_show = [c for c in ['product_name', 'brand', 'product_type', 'acne_types_str',
                              'acne_ingredient_count'] if c in acne_products.columns]
display(acne_products[cols_to_show].head(8))

Acne type tagging complete.

Acne type distribution:
  Comedonal           : 200 products
  Pigmentation        : 103 products
  Dark Circles        : 101 products
  Inflammatory        : 69 products
  Fungal              : 39 products
  Cystic              : 23 products
  General             : 16 products

Sample tagged products:


,product_name,product_type,acne_types_str,acne_ingredient_count
0,CeraVe Facial Moisturising Lotion SPF 25 52ml,Moisturiser,"Inflammatory, Pigmentation",1
1,AMELIORATE Transforming Body Lotion 200ml,Moisturiser,Comedonal,1
2,CeraVe Facial Moisturising Lotion No SPF 52ml,Moisturiser,"Inflammatory, Pigmentation",1
3,CeraVe Smoothing Cream 177ml,Moisturiser,"Comedonal, Inflammatory, Pigmentation",2
4,Clinique Moisture Surge 72 Hour Moisturiser 75ml,Moisturiser,Dark Circles,1
5,Clinique Dramatically Different Moisturising G...,Moisturiser,Dark Circles,1
6,La Roche-Posay Effaclar Duo+ SPF30 40ml,Moisturiser,"Inflammatory, Pigmentation",1
7,Estée Lauder DayWear Advanced Multi-Protection...,Moisturiser,Dark Circles,1


In [11]:
# ─────────────────────────────────────────────
# SECTION 5 — The Ingredient Matcher
#
# This is the bridge function.
# Input : ingredient formula from your recommender
#         e.g. 'Zinc PCA + Benzoyl Peroxide + Salicylic Acid'
# Output: top N real products ranked by ingredient overlap
# ─────────────────────────────────────────────

def parse_recommended_formula(formula_str):
    """
    Parse the recommender output formula into a list of ingredients.
    Handles formats like:
      'Zinc PCA + Benzoyl Peroxide + Salicylic Acid'
      'Zinc PCA(2.5%) + Benzoyl Peroxide(5%) + Niacinamide(1%)'
    """
    parts = formula_str.split('+')
    ingredients = []
    for part in parts:
        # Remove concentration annotations like (2.5%)
        clean = re.sub(r'\(.*?\)', '', part).strip().lower()
        if clean:
            ingredients.append(clean)
    return ingredients

def score_product(product_ingredients_list, recommended_ingredients):
    """
    Score a product based on how many recommended ingredients it contains.
    Returns a score between 0 and 1.

    Scoring:
    - +1.0 point for each recommended ingredient found (exact match)
    - +0.5 point for partial match (recommended ing appears inside a product ing name)
    - Final score = total points / number of recommended ingredients
    """
    if not recommended_ingredients or not product_ingredients_list:
        return 0.0

    total_score = 0.0
    matched = []

    for rec_ing in recommended_ingredients:
        found = False
        for prod_ing in product_ingredients_list:
            if rec_ing == prod_ing:             # exact match
                total_score += 1.0
                matched.append(rec_ing)
                found = True
                break
            elif rec_ing in prod_ing or prod_ing in rec_ing:  # partial match
                total_score += 0.5
                matched.append(f'{rec_ing} (~)')
                found = True
                break

    normalized_score = total_score / len(recommended_ingredients)
    return normalized_score, matched

def find_matching_products(recommended_formula, acne_type=None, top_n=3,
                           min_score=0.3, products_df=None):
    """
    Main bridge function.

    Args:
        recommended_formula : str — output from recommender
                              e.g. 'Zinc PCA(2.5%) + Salicylic Acid(1%) + Niacinamide(5%)'
        acne_type           : str — acne type to filter by (optional)
                              e.g. 'Comedonal', 'Inflammatory', 'Cystic'
        top_n               : int — number of products to return
        min_score           : float — minimum match score to include (0-1)
        products_df         : DataFrame — the acne products dataframe

    Returns:
        list of dicts with product info + match explanation
    """
    if products_df is None:
        products_df = acne_products

    # Parse the recommended formula
    rec_ingredients = parse_recommended_formula(recommended_formula)
    print(f'Searching for products containing: {rec_ingredients}')

    # Filter by acne type if specified
    if acne_type:
        pool = products_df[products_df['acne_types'].apply(
            lambda types: acne_type in types
        )].copy()
        print(f'Filtered to {len(pool)} products tagged for: {acne_type}')
    else:
        pool = products_df.copy()

    if pool.empty:
        print(f'No products found for acne type: {acne_type}. Searching all products.')
        pool = products_df.copy()

    # Score every product
    results = []
    for idx, row in pool.iterrows():
        score_result = score_product(row['ingredients_list'], rec_ingredients)
        if isinstance(score_result, tuple):
            score, matched_ings = score_result
        else:
            score, matched_ings = score_result, []

        if score >= min_score:
            # Build a name for display (handle different column names)
            name_col = next((c for c in ['product_name', 'name', 'Name', 'Product Name']
                           if c in row.index), row.index[0])
            brand_col = next((c for c in ['brand', 'Brand']
                            if c in row.index), None)
            type_col = next((c for c in ['product_type', 'type', 'Label', 'category']
                           if c in row.index), None)
            price_col = 'price_clean' if 'price_clean' in row.index else None

            results.append({
                'product_name'       : row[name_col],
                'brand'              : row[brand_col] if brand_col else 'Unknown',
                'product_type'       : row[type_col] if type_col else 'Unknown',
                'price'              : row[price_col] if price_col else None,
                'match_score'        : round(score, 3),
                'matched_ingredients': matched_ings,
                'all_acne_types'     : row['acne_types'],
                'acne_ingredient_count': row['acne_ingredient_count'],
            })

    # Sort by match score, then by number of acne ingredients (more = better)
    results.sort(key=lambda x: (x['match_score'], x['acne_ingredient_count']), reverse=True)
    return results[:top_n]

print('Ingredient matcher ready.')

Ingredient matcher ready.


In [12]:
# ─────────────────────────────────────────────
# SECTION 6 — Full Recommendation Pipeline
#             with Human-Readable Explanations
#
# This is the complete end-to-end function that
# SkinMeta will call after the CNN classifies
# the user's acne type.
# ─────────────────────────────────────────────

# Explanation templates — why each ingredient was chosen
INGREDIENT_EXPLANATIONS = {
    'salicylic acid'  : 'Salicylic acid is a BHA that penetrates pores and dissolves '
                        'the dead skin cells and excess oil causing blockages.',
    'benzoyl peroxide': 'Benzoyl peroxide kills acne-causing bacteria (C. acnes) directly '
                        'and reduces inflammation at the follicle.',
    'niacinamide'     : 'Niacinamide (Vitamin B3) regulates sebum production, reduces '
                        'redness, and strengthens the skin barrier.',
    'azelaic acid'    : 'Azelaic acid is antibacterial and anti-inflammatory, effective '
                        'against both acne and post-acne dark marks.',
    'zinc pca'        : 'Zinc PCA controls sebum production and has antimicrobial '
                        'properties that reduce comedone formation.',
    'retinol'         : 'Retinol (Vitamin A) accelerates cell turnover, prevents pore '
                        'clogging, and reduces acne scarring over time.',
    'glycolic acid'   : 'Glycolic acid exfoliates the skin surface to prevent '
                        'dead cell buildup that clogs pores.',
    'tea tree'        : 'Tea tree oil has natural antibacterial and anti-inflammatory '
                        'properties comparable to mild benzoyl peroxide.',
    'hyaluronic acid' : 'Hyaluronic acid hydrates without clogging pores, essential '
                        'for maintaining barrier health during acne treatment.',
    'caffeine'        : 'Caffeine constricts blood vessels and reduces puffiness, '
                        'making it effective for dark circles and inflammation.',
    'vitamin c'       : 'Vitamin C (ascorbic acid) is an antioxidant that neutralizes '
                        'free radicals from UV/pollution and fades dark spots.',
    'niacinamide'     : 'Niacinamide regulates sebum and brightens post-acne marks.',
    'kojic acid'      : 'Kojic acid inhibits melanin production to fade '
                        'post-inflammatory hyperpigmentation.',
    'alpha arbutin'   : 'Alpha arbutin is a gentle skin brightener that targets '
                        'dark spots without irritating acne-prone skin.',
    'tranexamic acid' : 'Tranexamic acid reduces pigmentation by blocking the '
                        'pathway that triggers excess melanin after inflammation.',
    'sulfur'          : 'Sulfur is keratolytic and antibacterial, particularly '
                        'effective against fungal and comedonal acne.',
    'zinc gluconate'  : 'Zinc gluconate reduces sebum and has proven anti-inflammatory '
                        'activity against inflammatory acne lesions.',
    'lactic acid'     : 'Lactic acid gently exfoliates and hydrates simultaneously, '
                        'suitable for sensitive acne-prone skin.',
    'mandelic acid'   : 'Mandelic acid is a large-molecule AHA that exfoliates slowly, '
                        'ideal for sensitive or darker skin tones with acne.',
    'adapalene'       : 'Adapalene is a synthetic retinoid that normalizes skin cell '
                        'turnover and prevents new comedone and cyst formation.',
    'ascorbic acid'   : 'Ascorbic acid (pure Vitamin C) is a potent antioxidant '
                        'that fights oxidative stress and brightens dark marks.',
}

# Environmental factor explanations (for the explainability requirement)
ENVIRONMENTAL_EXPLANATIONS = {
    'high_pollution' : 'High pollution levels increase oxidative stress on skin, '
                       'clogging pores and triggering inflammatory acne.',
    'high_humidity'  : 'High humidity increases sebum production and sweat, '
                       'creating an environment where acne bacteria thrive.',
    'high_uv'        : 'High UV index triggers inflammation and worsens '
                       'post-acne dark marks (PIH).',
    'low_humidity'   : 'Low humidity dehydrates skin, causing it to produce '
                       'excess oil as compensation, worsening acne.',
}


def generate_explanation(acne_type, severity, recommended_formula,
                          matched_products, environmental_factors=None):
    """
    Generate a full human-readable explanation for the recommendation.

    Args:
        acne_type           : str  — e.g. 'Comedonal'
        severity            : int  — 0 (mild) to 3 (very severe)
        recommended_formula : str  — from your dermatology recommender
        matched_products    : list — output of find_matching_products()
        environmental_factors: dict — e.g. {'pollution': 'high', 'uv': 'high'}
    """
    severity_labels = {0: 'mild', 1: 'moderate', 2: 'severe', 3: 'very severe'}
    sev_label = severity_labels.get(severity, 'moderate')

    lines = []
    lines.append('=' * 60)
    lines.append('  SKINMETA DERMAAI — PERSONALIZED RECOMMENDATION')
    lines.append('=' * 60)

    # --- Diagnosis summary ---
    lines.append(f'\nDiagnosis:')
    lines.append(f'  Acne type : {acne_type}')
    lines.append(f'  Severity  : {sev_label.title()} (level {severity}/3)')

    # --- Environmental context ---
    if environmental_factors:
        lines.append('\nEnvironmental factors detected:')
        if environmental_factors.get('pollution') == 'high':
            lines.append(f'  • {ENVIRONMENTAL_EXPLANATIONS["high_pollution"]}')
        if environmental_factors.get('humidity') == 'high':
            lines.append(f'  • {ENVIRONMENTAL_EXPLANATIONS["high_humidity"]}')
        if environmental_factors.get('humidity') == 'low':
            lines.append(f'  • {ENVIRONMENTAL_EXPLANATIONS["low_humidity"]}')
        if environmental_factors.get('uv') == 'high':
            lines.append(f'  • {ENVIRONMENTAL_EXPLANATIONS["high_uv"]}')

    # --- Recommended ingredient formula ---
    rec_ingredients = parse_recommended_formula(recommended_formula)
    lines.append(f'\nRecommended treatment formula:')
    lines.append(f'  {recommended_formula}')
    lines.append('\nWhy these ingredients:')
    for ing in rec_ingredients:
        explanation = INGREDIENT_EXPLANATIONS.get(ing, f'{ing.title()} targets your acne concern.')
        lines.append(f'  • {ing.title()}: {explanation}')

    # --- Product recommendations ---
    lines.append(f'\nTop {len(matched_products)} recommended products:')
    for i, product in enumerate(matched_products, 1):
        lines.append(f'\n  [{i}] {product["product_name"]}')
        lines.append(f'      Brand        : {product["brand"]}')
        lines.append(f'      Type         : {product["product_type"]}')
        if product['price']:
            lines.append(f'      Price        : £{product["price"]:.2f}')
        lines.append(f'      Match score  : {product["match_score"] * 100:.0f}%')
        lines.append(f'      Contains     : {", ".join(product["matched_ingredients"])}')
        lines.append(f'      Good for     : {", ".join(product["all_acne_types"])}')

    lines.append('\n' + '=' * 60)
    return '\n'.join(lines)


def skinmeta_recommend(acne_type, severity, recommended_formula,
                       skin_profile=None, environmental_factors=None,
                       top_n=3):
    """
    MASTER FUNCTION — called after CNN classifies user's photo.

    Args:
        acne_type            : str  — CNN output, e.g. 'Comedonal'
        severity             : int  — CNN output, 0-3
        recommended_formula  : str  — from dermatology recommender
        skin_profile         : dict — user inputs, e.g.
                               {'age_group': '14-18',
                                'skin_type': 'Oily',
                                'sensitivity': 'Yes'}
        environmental_factors: dict — e.g.
                               {'pollution': 'high',
                                'humidity': 'high',
                                'uv': 'moderate'}
        top_n                : int  — number of products to return

    Returns:
        dict with 'products' list and 'explanation' string
    """
    print(f'Running SkinMeta recommendation...')
    print(f'  Acne type : {acne_type}  |  Severity : {severity}')
    print(f'  Formula   : {recommended_formula}')

    # Step 1: find matching products
    matched = find_matching_products(
        recommended_formula=recommended_formula,
        acne_type=acne_type,
        top_n=top_n
    )

    if not matched:
        # Fallback: search without acne type filter
        print('  No exact matches. Broadening search...')
        matched = find_matching_products(
            recommended_formula=recommended_formula,
            acne_type=None,
            top_n=top_n
        )

    # Step 2: generate explanation
    explanation = generate_explanation(
        acne_type=acne_type,
        severity=severity,
        recommended_formula=recommended_formula,
        matched_products=matched,
        environmental_factors=environmental_factors
    )

    return {
        'acne_type'   : acne_type,
        'severity'    : severity,
        'formula'     : recommended_formula,
        'products'    : matched,
        'explanation' : explanation
    }

print('Full recommendation pipeline ready.')

Full recommendation pipeline ready.


In [13]:
# ─────────────────────────────────────────────
# SECTION 6B — Test the Full Pipeline
#
# Simulating what happens after the CNN runs on
# a user's photo and the recommender picks a formula
# ─────────────────────────────────────────────

# --- Test Case 1: Comedonal acne, teenage oily skin ---
print('\n>>> TEST CASE 1')
result1 = skinmeta_recommend(
    acne_type='Comedonal',
    severity=1,
    recommended_formula='Zinc PCA(2.5%) + Benzoyl Peroxide(2%) + Salicylic Acid(1%)',
    skin_profile={
        'age_group' : '14-18',
        'skin_type' : 'Oily',
        'sensitivity': 'No'
    },
    environmental_factors={
        'pollution': 'high',
        'humidity' : 'high',
        'uv'       : 'moderate'
    }
)
print(result1['explanation'])

print('\n\n>>> TEST CASE 2')
# --- Test Case 2: Inflammatory acne, adult dry skin ---
result2 = skinmeta_recommend(
    acne_type='Inflammatory',
    severity=2,
    recommended_formula='Benzoyl Peroxide(2.5%) + Niacinamide(5%) + Azelaic Acid(10%)',
    skin_profile={
        'age_group' : '25-35',
        'skin_type' : 'Dry',
        'sensitivity': 'Yes'
    },
    environmental_factors={
        'pollution': 'moderate',
        'humidity' : 'low',
        'uv'       : 'high'
    }
)
print(result2['explanation'])


>>> TEST CASE 1
Running SkinMeta recommendation...
  Acne type : Comedonal  |  Severity : 1
  Formula   : Zinc PCA(2.5%) + Benzoyl Peroxide(2%) + Salicylic Acid(1%)
Searching for products containing: ['zinc pca', 'benzoyl peroxide', 'salicylic acid']
Filtered to 200 products tagged for: Comedonal
  SKINMETA DERMAAI — PERSONALIZED RECOMMENDATION

Diagnosis:
  Acne type : Comedonal
  Severity  : Moderate (level 1/3)

Environmental factors detected:
  • High pollution levels increase oxidative stress on skin, clogging pores and triggering inflammatory acne.
  • High humidity increases sebum production and sweat, creating an environment where acne bacteria thrive.

Recommended treatment formula:
  Zinc PCA(2.5%) + Benzoyl Peroxide(2%) + Salicylic Acid(1%)

Why these ingredients:
  • Zinc Pca: Zinc PCA controls sebum production and has antimicrobial properties that reduce comedone formation.
  • Benzoyl Peroxide: Benzoyl peroxide kills acne-causing bacteria (C. acnes) directly and reduces 

In [14]:
# ─────────────────────────────────────────────
# SECTION 7 — Save All Outputs
# ─────────────────────────────────────────────
import pickle

# 1. Save filtered + tagged product dataset
save_cols = [c for c in [
    'product_name', 'brand', 'product_type', 'price_clean',
    ing_col, 'acne_ingredients_present', 'acne_ingredient_count',
    'acne_types', 'acne_types_str'
] if c in acne_products.columns]

acne_products[save_cols].to_csv('acne_products_tagged.csv', index=False)
print('Saved: acne_products_tagged.csv')

# 2. Save the ingredient explanations dict (for explainability module)
with open('ingredient_explanations.pkl', 'wb') as f:
    pickle.dump(INGREDIENT_EXPLANATIONS, f)
print('Saved: ingredient_explanations.pkl')

# 3. Save acne type rules (for documentation)
with open('acne_type_rules.json', 'w') as f:
    json.dump(ACNE_TYPE_RULES, f, indent=2)
print('Saved: acne_type_rules.json')

# 4. Save a sample recommendation as JSON (for report)
result_for_json = {
    'acne_type': result1['acne_type'],
    'severity' : result1['severity'],
    'formula'  : result1['formula'],
    'products' : result1['products'],
}
with open('sample_recommendation_output.json', 'w') as f:
    json.dump(result_for_json, f, indent=2)
print('Saved: sample_recommendation_output.json')

# Final summary
print('\n' + '=' * 55)
print('  BRIDGE LAYER SUMMARY')
print('=' * 55)
print(f'  Acne-relevant products : {len(acne_products)}')
print(f'  Acne types covered     : {list(ACNE_TYPE_RULES.keys())}')
print(f'  Ingredients tracked    : {len(ACNE_INGREDIENTS)}')
print(f'  Explanations available : {len(INGREDIENT_EXPLANATIONS)}')
print('='*55)
print('  Ready for Phase 3 model integration')
print('=' * 55)

Saved: acne_products_tagged.csv
Saved: ingredient_explanations.pkl
Saved: acne_type_rules.json
Saved: sample_recommendation_output.json

  BRIDGE LAYER SUMMARY
  Acne-relevant products : 382
  Acne types covered     : ['Comedonal', 'Inflammatory', 'Cystic', 'Fungal', 'Pigmentation', 'Dark Circles']
  Ingredients tracked    : 22
  Explanations available : 20
  Ready for Phase 3 model integration


---
## How this connects to the rest of the system

```
User uploads photo
       ↓
CNN classifies → acne_type='Comedonal', severity=1
       ↓
Recommender queries dermatology_skincare.csv
       → formula = 'Zinc PCA(2.5%) + Salicylic Acid(1%)'
       ↓
skinmeta_recommend() ← THIS NOTEBOOK
       → searches acne_products_tagged.csv
       → scores by ingredient overlap
       → generates human-readable explanation
       ↓
User sees: top 3 products + why each ingredient was chosen
```

## Files saved
| File | Purpose |
|---|---|
| `acne_products_tagged.csv` | Filtered product dataset with acne type tags |
| `ingredient_explanations.pkl` | Explanations for each ingredient |
| `acne_type_rules.json` | Rules used to assign acne type tags |
| `sample_recommendation_output.json` | Example output for LaTeX report |